In [1]:
import gridlabd

In [2]:
dir(gridlabd)

['ConfigurationError',
 'GLDCheckPointMode',
 'GLDErrorCode',
 'GridLABDError',
 'GridLabD',
 'Path',
 'Simulation',
 'SimulationError',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '_lib_dir',
 '_package_dir',
 '_share_dir',
 'build_lib_dir',
 'bundle_utils',
 'get_gridlabd_info',
 'gldcore_dir',
 'glpath',
 'glpath_components',
 'gridlabd_core',
 'hello',
 'info',
 'load_model',
 'os',
 'path_sep',
 'repo_root',
 'root',
 'setup_bundled_environment',
 'simulation',
 'version']

In [ ]:
gld = gridlabd.GridLabD()

In [ ]:
from pathlib import Path
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld.set_working_directory(str(model_dir))


In [ ]:
gld.set_config_file("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/gridlabd.conf")

In [ ]:
gld.load_glm(["gridlabd", "./test_HVAC_balance.glm", "--verbose"])

In [ ]:
gld.run()

In [ ]:
# Don't call exit_gld() in notebooks - it crashes the kernel
# Just let Python clean up automatically
del gld

## Testing More Functionality

In [3]:
# Create a new instance for testing
import json
from pathlib import Path
gld2 = gridlabd.GridLabD()
model_dir = Path("/mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests/test_HVAC_balance.glm").parent
gld2.set_working_directory(str(model_dir))


Working directory set to: /mnt/c/Projects/Gridlab-d/gridlab-d/python_bindings/tests


GLDErrorCode.SUCCESS

In [ ]:
del gld2

In [4]:

gld2.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

GLDErrorCode.SUCCESS

In [5]:
gld2.run()


Using previous start_time: 0.00
Using previous stop_time: 0.00

WARNING  [INIT] : Daylight saving time (DST) is not handled correctly when using TMY2 datasets; please use TMY3 for DST-corrected weather data.


GLDErrorCode.SUCCESS

In [6]:

# Get checkpoint as JSON string
checkpoint_json = gld2.get_checkpoint_json()
checkpoint_data = json.loads(checkpoint_json)

print("Checkpoint keys:", list(checkpoint_data.keys())[:10])  # First 10 keys
print(f"Total objects: {len(checkpoint_data)}")

Checkpoint keys: ['__preamble', 'clock', 'objects']
Total objects: 3


In [7]:
list(checkpoint_data.keys())

['__preamble', 'clock', 'objects']

In [13]:
checkpoint_data['__preamble']

{'comments': ['// GridLAB-D checkpoint data export',
  '// Generated at timestamp: 989150402',
  '// Checkpoint sequence: 0']}

In [14]:
checkpoint_data['clock']

{'starttime': 989064000,
 'stoptime': 989150402,
 'timestamp': 989150402,
 'timezone': 'PST8PDT'}

In [17]:
list(checkpoint_data['objects'])

['climate', 'house', 'recorder', 'triplex_meter']

In [20]:
# Get current simulation time
status, current_time = gld2.get_time()
print(f"Status: {status}")
print(f"Current simulation time: {current_time}")

Status: GLDErrorCode.SUCCESS
Current simulation time: 2025-06-12T12:00:00
Getting current time: 2025-06-12T12:00:00


In [ ]:
gld2.step()

: 

In [21]:

# Get checkpoint as JSON string...again
checkpoint_json2 = gld2.get_checkpoint_json()
checkpoint_data2 = json.loads(checkpoint_json2)

print("Checkpoint keys:", list(checkpoint_data2.keys())[:10])  # First 10 keys
print(f"Total objects: {len(checkpoint_data2)}")

AttributeError: 'NoneType' object has no attribute 'keys'

In [22]:
# Debug: Check what checkpoint_json2 actually contains
print("checkpoint_json2 type:", type(checkpoint_json2))
print("checkpoint_json2 length:", len(checkpoint_json2))
print("checkpoint_json2 content (first 200 chars):", checkpoint_json2[:200])
print("checkpoint_json2 content (last 200 chars):", checkpoint_json2[-200:])

checkpoint_json2 type: <class 'str'>
checkpoint_json2 length: 4
checkpoint_json2 content (first 200 chars): null
checkpoint_json2 content (last 200 chars): null


In [ ]:
# Check checkpoint configuration (these are global variables in GridLAB-D)
# You can access them via the GLM command line arguments or config

# For now, the simplest solution is to:
# 1. Call get_checkpoint_json() once after run() and save the result
# 2. Reuse that data instead of calling again

# OR advance the simulation time significantly between calls:
# - For CPT_SIM (default): advance simulation by 86400+ seconds (1 day)
# - For CPT_WALL: wait 3600+ real seconds (not practical!)

print("Best practice: Cache the first checkpoint result and reuse it")
print("The checkpoint data doesn't change unless you run/step the simulation further")

### Understanding Checkpoint Intervals

GridLAB-D uses global variables to control checkpoint behavior:
- `checkpoint_type`: NONE (0), WALL (1), or SIM (2)
- `checkpoint_interval`: Time between checkpoints (seconds for WALL, simulation seconds for SIM)
  - Default for WALL: 3600 seconds (1 hour)
  - Default for SIM: 86400 seconds (1 day simulation time)

In [ ]:
# Run simulation step by step
gld3 = gridlabd.GridLabD()
gld3.set_working_directory(str(model_dir))
gld3.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

print("Starting step-by-step simulation...")
for i in range(5):  # Run 5 steps
    sim_time = gld3.step()
    status, time_str = gld3.get_time()
    print(f"Step {i+1}: time = {sim_time} ({time_str})")
    
print("Step simulation complete")

In [ ]:
# Use the higher-level Simulation wrapper
from gridlabd import Simulation

with Simulation() as sim:
    sim.gld.set_working_directory(str(model_dir))
    result = sim.gld.load_glm(["gridlabd", "./test_HVAC_balance.glm"])
    print(f"Load result: {result}")
    
    result = sim.gld.run()
    print(f"Run result: {result}")
    
    # Get checkpoint data through the wrapper
    checkpoint = sim.gld.get_checkpoint_json()
    data = json.loads(checkpoint)
    print(f"Objects in simulation: {len(data)}")

In [ ]:
# Get installation paths
print("GridLAB-D Info:")
print(f"  Version: {gridlabd.version()}")
print(f"  Install root: {gridlabd.GridLabD.get_install_root()}")
print(f"  Executable: {gridlabd.GridLabD.get_executable_path()}")
print(f"  Hello: {gridlabd.hello()}")

In [ ]:
# Run with specific start/stop times
gld4 = gridlabd.GridLabD()
gld4.set_working_directory(str(model_dir))
gld4.load_glm(["gridlabd", "./test_HVAC_balance.glm"])

# Run for a specific time range (example timestamps)
# Note: times are in seconds since epoch
result = gld4.run(start_time=0.0, stop_time=3600.0)  # Run for 1 hour
print(f"Simulation result: {result}")

status, final_time = gld4.get_time()
print(f"Final time: {final_time}")

In [ ]:
# NOTE: get_checkpoint_json() uses an internal checkpoint interval
# Calling it twice rapidly may return empty data the second time
# Solution: Either reuse the first data or pass "" to force a fresh checkpoint

# Get checkpoint with explicit empty path to force fresh data
checkpoint_json2 = gld2.get_checkpoint_json("")
checkpoint_data2 = json.loads(checkpoint_json2) if checkpoint_json2 else None

if checkpoint_data2:
    print("Checkpoint keys:", list(checkpoint_data2.keys())[:10])  # First 10 keys
    print(f"Total objects: {len(checkpoint_data2)}")
else:
    print("No checkpoint data returned (interval not met). Reusing original checkpoint_data.")
    checkpoint_data2 = checkpoint_data
if load_result == gridlabd.GLDErrorCode.SUCCESS:
    run_result = gld5.run()
    print(f"Run result: {run_result}")
    
    # Get some data
    checkpoint = gld5.get_checkpoint_json()
    data = json.loads(checkpoint)
    print(f"Successfully simulated {len(data)} objects")

In [ ]:
# Analyze checkpoint data with pandas (if available)
try:
    import pandas as pd
    
    # Convert checkpoint to DataFrame for analysis
    checkpoint_json = gld5.get_checkpoint_json()
    data = json.loads(checkpoint_json)
    
    # Create a summary DataFrame
    summary = []
    for obj_name, obj_data in list(data.items())[:20]:  # First 20 objects
        summary.append({
            'name': obj_name,
            'class': obj_data.get('class', 'unknown'),
            'parent': obj_data.get('parent', 'none'),
            'rank': obj_data.get('rank', -1)
        })
    
    df = pd.DataFrame(summary)
    print("Object Summary:")
    print(df)
    print(f"\nClass distribution:\n{df['class'].value_counts()}")
    
except ImportError:
    print("pandas not available, skipping DataFrame analysis")

### 8. Analyzing Checkpoint Data with Pandas

### 7. Using the Convenience Function

### 6. Running with Custom Start/Stop Times

### 5. Query GridLAB-D Installation Info

### 4. Using the High-Level Simulation Class

### 3. Step-by-Step Simulation

### 2. Query Simulation Time

In [ ]:
# Inspect a specific object from the checkpoint
if len(checkpoint_data) > 0:
    first_obj_name = list(checkpoint_data.keys())[0]
    print(f"\nFirst object: {first_obj_name}")
    print(json.dumps(checkpoint_data[first_obj_name], indent=2)[:500])  # First 500 chars

### 1. Get Checkpoint JSON (Model State)